# Qwen3-Next Mini

Hybrid gated linear attention + periodic full attention.

## Setup

In [1]:
from pathlib import Path
import sys

PACKAGES = Path.cwd()
while not (PACKAGES / "qwen_common").exists():
    if PACKAGES == PACKAGES.parent:
        raise RuntimeError("Run this notebook from inside deepseek_flash_lab")
    PACKAGES = PACKAGES.parent
sys.path.insert(0, str(PACKAGES))

import torch

torch.manual_seed(0)

## Hybrid layer pattern

In [2]:
from qwen3_next_mini import Qwen3NextModel, preset

cfg = preset("pc")
print("layer pattern:", cfg.layer_pattern)
model = Qwen3NextModel(cfg)
tokens = torch.randint(0, cfg.vocab_size, (2, 8))
logits, _ = model(tokens)
print("logits:", tuple(logits.shape), "params:", f"{model.num_parameters():,}")

layer pattern: ['linear', 'linear', 'linear', 'full']
logits: (2, 8, 64) params: 953,696


## Causality check

Changing the last token must not change earlier outputs.

In [3]:
model.eval()
a, _ = model(tokens)
changed = tokens.clone()
changed[0, -1] = (changed[0, -1] + 1) % cfg.vocab_size
b, _ = model(changed)
print("earlier diff:", (a[0, :-1] - b[0, :-1]).abs().max().item())
print("last diff   :", (a[0, -1] - b[0, -1]).abs().max().item())

earlier diff: 9.685754776000977e-08
last diff   : 1.4460049867630005


## Checkpoint

- Why is the linear recurrence O(T) instead of O(T²)?
- Which layers keep full attention, and why keep any?